In [ ]:
# Install dependencies:
# pip install --upgrade langchain==0.3.0 langchain-community langchain-openai langchain-chroma langchain-huggingface sentence-transformers unstructured pypdf pdfplumber python-dotenv transformers

import os
import logging
import shutil
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from openai import RateLimitError, APIConnectionError
from transformers import AutoTokenizer
import time
from langchain.cache import InMemoryCache

In [ ]:
load_dotenv()

In [ ]:
#absolute path
#robust path handling for code portability(script_dir and os.path.join) to avoid hardcoding absolute paths that might break on other machines if you move your script
DATA_DIR="C:/Users/USER/Documents/DS_LUX/Finance_Bill_259(RAG).pdf"
VECTOR_DB_DIR="chroma_db/"

In [ ]:
logging.basicConfig(level=logging.INFO)
logger=logging.getLogger(__name__)

In [ ]:
EMBEDDING_MAX_TOKENS = 256
CHUNK_SIZE_TOKENS = 200 # A bit less than 256 to give buffer
CHUNK_OVERLAP_TOKENS = 50 

In [ ]:

def ingest_data():
    logger.info(f"Loading documents from {DATA_DIR}...")
    if not os.path.exists(DATA_DIR):
        logger.error(f"PDF file not found at {DATA_DIR}")
        raise FileNotFoundError(f"PDF not found at {DATA_DIR}")
    
    try:
        loader=PyPDFLoader("C:/Users/USER/Documents/DS_LUX/Finance_Bill_259(RAG).pdf")
        documents=loader.load()
        logger.info(f"Loaded {len(documents)} documents successfuly...")
    except Exception as e:
        logger.error(f"Failed to load PDF: {e}")
        raise

    logger.info("Splitting documents into chunks...")#splitting into chunks.
    #token-based splitting to respect model limits
    tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
    def token_length(text):
        return len(tokenizer.encode(text, add_special_tokens=True))#defining token length fxn based on the embedding tokenizer
    
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE_TOKENS, #max size of each chunk
        chunk_overlap=CHUNK_OVERLAP_TOKENS, #overlap between chunks to maintain context
        length_function=token_length, #custom fxn
        is_separator_regex=False, #standard separators
    )

    chunks=text_splitter.split_documents(documents)
    logger.info(f"Split into {len(chunks)} chunks successfully.")

    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2",
                                       model_kwargs={"device":"cpu"},#were GPU available, "cuda" would be well-suited
                                       encode_kwargs={"normalize_embeddings":True,
                                                        "truncate": True, #ensure truncation if a chunk somehow still exceeds
                                                        "max_length":EMBEDDING_MAX_TOKENS,
                                                        "batch_size":32})#truncate to 512 texts


    #embeddings creation and vector db storage.
    logger.info("Creating embeddings and instoring to ChromaDB...")

    #create ChromaDB vector store to load the database in the specified directory.
    os.makedirs(VECTOR_DB_DIR, exist_ok=True)

    try:
        valid_chunks = []
        for chunk in chunks:
            tokens = token_length(chunk.page_content)
            if tokens > EMBEDDING_MAX_TOKENS:
                logger.warning(f"Truncating chunk with {tokens} tokens")
                chunk.page_content = tokenizer.decode(tokenizer.encode(chunk.page_content, max_length=256, truncation=True))
            valid_chunks.append(chunk)

        vectorstore=Chroma.from_documents(
            documents=chunks,
            embedding=embeddings,
            persist_directory=VECTOR_DB_DIR
        )
        logger.info(f"Embeddings stored in {VECTOR_DB_DIR}")
        logger.info(f"Number of documents in vector store is:",vectorstore._collection.count())
        if vectorstore._collection.count()==0:
            logger.error("No documents stored in ChromaDB")
            raise ValueError("Ingestion failed: No documents stored")
    except Exception as e:
        logger.error(f"Failed to store embeddings: {e}")
        raise

    if vectorstore._collection.count()>0:
        print(f"Ingestion step successful to {VECTOR_DB_DIR} directory successfully...")
    else:
        print("Ingestion step failed to store embeddings.ERROR!")


In [ ]:

VECTOR_DB_DIR="chroma_db/" #directory where the ChromaDB is stored.

def setup_rag_chain():
    logger.info(f"Loading vector db from {VECTOR_DB_DIR}...")
    
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={"device":"cpu"},
        encode_kwargs={"normalize_embedings":True,
                       "truncate":True,
                       "max_length":EMBEDDING_MAX_TOKENS,"batch_size":32}#same embeddings model same as that for ingestion.
    )

    vectorstore=Chroma(
        embedding_function=embeddings,
        persist_directory=VECTOR_DB_DIR
    )#loading the existing chromaDB, note the chunks and their respective embeddings form the chroma.

    count= vectorstore._collection.count()
    logger.info(f"Vector store contains {count} documents")
    if count==0:
        logger.error("Vector store is empty! Ingest data first.")
        raise ValueError("Chroma database is empty.!")
    
    retriever=vectorstore.as_retriever(search_kwargs={"k":1})#retrieve the top n relevant chunks.

    logger.info("Initializing LLM...")#now initializing the llm
    llm=ChatOpenAI(model="gpt-3.5-turbo",temperature=0.3,max_completion_tokens=200)#temp fpr randomness, higher more creative, more random.#model for for chat and text completion.
    #llm=HuggingFacePipeline.from_model_id(model_id="sentence-transformers/all-MiniLM-L6-v2")

    template="""I am an AI assistant that answers your questions off of The Proposed Finance Bill 25/26.
    context:{context}
    question:{question}"""
    prompt=ChatPromptTemplate.from_template(template)#prompt for context and question.

    rag_chain=(
        {"context":retriever, "question":RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    logger.info("RAG chain setup complete.")
    return rag_chain


def chat_with_rag(rag_chain):
    logger.info("Chatbot ready! Type 'exit' to quit.")
    max_retries=3
    retry_delay=60 #seconds

    while True:
        user_input=input("You: ")
        if user_input.lower()=="exit":
            print("Goodbye!")
            break

        print("Thinkiinggg....")
        for attempt in range(max_retries):
            try:
                response=rag_chain.invoke(user_input)
                print(f"CHAT: {response}")
                break #break out of the retry loop on success

            except Exception as e:
                if isinstance(e, (RateLimitError, APIConnectionError)): # Check for specific OpenAI errors
                    if attempt < max_retries - 1:
                        print(f"OpenAI API error: {e}. Retrying in {retry_delay} seconds...")
                        time.sleep(retry_delay)
                        retry_delay *= 1.5
                else:
                    print(f"Max retries reached for OpenAI API. Last error: {e}")
                    return # to stop trying this query.
            except Exception as e:
                logger.error(f"An unexpected error occurred: {e}")#Decide if you want to retry for other errors or break
                if "Token indices sequence length is longer than" in str(e) or "The expanded size of the tensor" in str(e):
                    print(f"Error: Input sequence too long for the model: {e}")
                    print("Please check chunk sizes or model configuration.")
                    break
                else:
                    print(f"An unexpected error occurred:{e}")
                    break

In [ ]:
if __name__=="__main__":

    #clear existing ChromaDB
    if os.path.exists(VECTOR_DB_DIR):
        shutil.rmtree(VECTOR_DB_DIR)
        logger.info("Cleared existing Chroma DB")
    
    #run ingestion
    try:
        logger.info("Running ingestion")
        ingest_data()
    except Exception as e:
        logger.error(f"Error ingesting data: {e}")
        raise

    #run RAG chain
    try:
        rag_chain=setup_rag_chain()
        chat_with_rag(rag_chain)
    except Exception as e:
        logger.error(f"Setup Error: {e}.Please ensure your db is properly ingested.!")
        raise